In [1]:
import gc
import os
import psutil

os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
gc.collect()

ram = psutil.virtual_memory()
print(f"💾 Total RAM:     {ram.total/1024**3:.1f} GB")
print(f"💾 Available RAM: {ram.available/1024**3:.1f} GB")
print(f"💾 Used RAM:      {ram.used/1024**3:.1f} GB")
print(f"\n{'✅ OK to proceed!' if ram.available > 2*1024**3 else '⚠️ Close more apps first!'}")

💾 Total RAM:     15.3 GB
💾 Available RAM: 11.2 GB
💾 Used RAM:      4.1 GB

✅ OK to proceed!


In [2]:
import gc, os, json, pickle, warnings
import numpy as np
import pandas as pd
import faiss
import psutil
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
tqdm.pandas()
warnings.filterwarnings('ignore')

os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

DATA_DIR = '/home/arshad/Network-project/data/'

DTYPE_MAP = {
    'Destination Port':       'int32',  'Flow Duration':          'float32',
    'Total Fwd Packets':      'int32',  'Total Backward Packets': 'int32',
    'Fwd Packet Length Mean': 'float32','Bwd Packet Length Mean': 'float32',
    'Packet Length Mean':     'float32','Packet Length Std':      'float32',
    'Flow Bytes/s':           'float32','Flow Packets/s':         'float32',
    'Fwd Packets/s':          'float32','Bwd Packets/s':          'float32',
    'Flow IAT Mean':          'float32','Flow IAT Std':           'float32',
    'SYN Flag Count':         'int16',  'ACK Flag Count':         'int16',
    'FIN Flag Count':         'int16',  'RST Flag Count':         'int16',
    'PSH Flag Count':         'int16',  'URG Flag Count':         'int16',
    'Init_Win_bytes_forward': 'int32',  'Init_Win_bytes_backward':'int32',
    'Active Mean':            'float32','Idle Mean':              'float32',
    'Down/Up Ratio':          'float32','Average Packet Size':    'float32',
    'pred_confidence_1':      'float32','pred_confidence_2':      'float32',
    'pred_confidence_3':      'float32',
}

feature_cols = [
    'Destination Port', 'Flow Duration',
    'Total Fwd Packets', 'Total Backward Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Packet Length Mean', 'Packet Length Std',
    'Flow Bytes/s', 'Flow Packets/s',
    'Fwd Packets/s', 'Bwd Packets/s',
    'Flow IAT Mean', 'Flow IAT Std',
    'SYN Flag Count', 'ACK Flag Count',
    'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count',
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
    'Active Mean', 'Idle Mean',
    'Down/Up Ratio', 'Average Packet Size'
]

tactic_map = {
    'BENIGN':                    'benign',
    'DoS Hulk':                  'impact',
    'DoS GoldenEye':             'impact',
    'DoS slowloris':             'impact',
    'DoS Slowhttptest':          'impact',
    'DDoS':                      'impact',
    'PortScan':                  'discovery',
    'FTP-Patator':               'credential-access',
    'SSH-Patator':               'credential-access',
    'Bot':                       'command-and-control',
    'Web Attack - Brute Force':  'initial-access',
    'Web Attack - XSS':          'initial-access',
    'Web Attack - SQL Injection':'initial-access',
    'Heartbleed':                'initial-access',
    'Infiltration':              'lateral-movement',
}

print("✅ Imports done")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")

✅ Imports done
💾 RAM available: 10.9 GB


In [3]:
print("⏳ Loading data...")

LOAD_COLS = list(DTYPE_MAP.keys()) + [
    'Label', 'MITRE_Tactic', 'MITRE_Technique', 'MITRE_Tech_Name',
    'alert_text',
    'pred_technique_1', 'pred_tech_name_1', 'pred_tactic_1',
    'pred_technique_2', 'pred_tech_name_2', 'pred_tactic_2',
    'pred_technique_3', 'pred_tech_name_3', 'pred_tactic_3',
]

df = pd.read_csv(
    DATA_DIR + 'sample_with_mitre_matches.csv',
    usecols=LOAD_COLS,
    dtype=DTYPE_MAP,
    low_memory=False
)
gc.collect()

# Fix NaN
df['MITRE_Tactic']    = df['MITRE_Tactic'].fillna('None')
df['MITRE_Technique'] = df['MITRE_Technique'].fillna('None')
df['MITRE_Tech_Name'] = df['MITRE_Tech_Name'].fillna('None')
df['tactic_label']    = df['Label'].map(tactic_map)

# Smart sampling — keeps all rare classes, caps large ones
sample_limits = {
    'BENIGN':                    5000,
    'DoS Hulk':                 10000,
    'DDoS':                     10000,
    'PortScan':                 10000,
    'DoS GoldenEye':             5000,
    'FTP-Patator':               5000,
    'DoS slowloris':             5000,
    'DoS Slowhttptest':          5000,
    'SSH-Patator':               3000,
    'Bot':                       1953,
    'Web Attack - Brute Force':  1470,
    'Web Attack - XSS':           652,
    'Infiltration':                36,
    'Web Attack - SQL Injection':  21,
    'Heartbleed':                  11,
}

samples = []
for label, limit in sample_limits.items():
    subset = df[df['Label'] == label]
    n      = min(len(subset), limit)
    samples.append(subset.sample(n, random_state=42))

df = pd.concat(samples, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
gc.collect()

mem_mb    = df.memory_usage(deep=True).sum() / 1024**2
ram_avail = psutil.virtual_memory().available / 1024**3

print(f"✅ Loaded: {len(df):,} rows × {df.shape[1]} columns")
print(f"💾 DataFrame RAM:        {mem_mb:.0f} MB")
print(f"💾 System RAM available: {ram_avail:.1f} GB")
print(f"\n📊 Label distribution:")
print(df['Label'].value_counts().to_string())

⏳ Loading data...
✅ Loaded: 62,143 rows × 44 columns
💾 DataFrame RAM:        78 MB
💾 System RAM available: 10.5 GB

📊 Label distribution:
Label
DoS Hulk                      10000
DDoS                          10000
PortScan                      10000
FTP-Patator                    5000
BENIGN                         5000
DoS Slowhttptest               5000
DoS slowloris                  5000
DoS GoldenEye                  5000
SSH-Patator                    3000
Bot                            1953
Web Attack - Brute Force       1470
Web Attack - XSS                652
Infiltration                     36
Web Attack - SQL Injection       21
Heartbleed                       11


In [4]:
def generate_alert_text(row):
    dst_port      = int(row.get('Destination Port', 0))
    syn           = int(row.get('SYN Flag Count', 0))
    ack           = int(row.get('ACK Flag Count', 0))
    fin           = int(row.get('FIN Flag Count', 0))
    rst           = int(row.get('RST Flag Count', 0))
    psh           = int(row.get('PSH Flag Count', 0))
    fwd_pkts      = int(row.get('Total Fwd Packets', 0))
    bwd_pkts      = int(row.get('Total Backward Packets', 0))
    pkt_rate      = float(row.get('Flow Packets/s', 0))
    pkt_len       = float(row.get('Packet Length Mean', 0))
    byte_rate     = float(row.get('Flow Bytes/s', 0))
    duration      = float(row.get('Flow Duration', 0)) / 1e6
    iat_mean      = float(row.get('Flow IAT Mean', 0))
    down_up       = float(row.get('Down/Up Ratio', 0))
    flow_pkts     = fwd_pkts + bwd_pkts
    fwd_bwd_ratio = fwd_pkts / (bwd_pkts + 1)

    port_context = {
        80:'HTTP web server',   443:'HTTPS web server',
        22:'SSH remote access', 21:'FTP file transfer server',
        23:'Telnet service',    3389:'RDP remote desktop',
        53:'DNS resolver',      3306:'MySQL database server',
        8080:'HTTP web proxy',  445:'SMB file sharing',
    }
    port_desc = port_context.get(dst_port, f'port {dst_port}')
    flags     = [n for f,n in [(syn,'SYN'),(ack,'ACK'),(fin,'FIN'),
                               (rst,'RST'),(psh,'PSH')] if f]
    flag_str  = ', '.join(flags) if flags else 'no TCP flags'

    if pkt_len < 20 and flow_pkts <= 3 and pkt_rate > 1000:
        return (
            f"Network reconnaissance and port scanning probing {port_desc}. "
            f"Adversary enumerating open network services to discover attack surface. "
            f"Tiny {pkt_len:.1f} byte probes at {pkt_rate:.0f} packets/sec with {flag_str}. "
            f"Consistent with network service discovery and host enumeration."
        )
    if dst_port in [21, 22, 23, 3389, 5900]:
        svc = {21:'FTP',22:'SSH',23:'Telnet',3389:'RDP',5900:'VNC'}.get(dst_port,'remote')
        return (
            f"Credential brute force against {svc} on {port_desc}. "
            f"Adversary systematically guessing passwords for unauthorized access. "
            f"{fwd_pkts} fwd and {bwd_pkts} bwd packets over {duration:.2f}s "
            f"at {pkt_rate:.1f} pkt/s with {flag_str}. "
            f"Consistent with brute force and password spraying credential access."
        )
    if pkt_len > 400 and pkt_rate > 50 and fwd_bwd_ratio > 2:
        return (
            f"Distributed denial of service flood targeting {port_desc}. "
            f"High volume {pkt_len:.0f} byte packets at {pkt_rate:.0f} pkt/s "
            f"consuming {byte_rate:.0f} bytes/sec bandwidth. "
            f"{fwd_pkts} fwd vs {bwd_pkts} bwd with {flag_str}. "
            f"Consistent with network denial of service and direct network flood."
        )
    if pkt_rate > 200 and fwd_bwd_ratio > 3:
        return (
            f"Denial of service flood exhausting resources at {port_desc}. "
            f"Sending {pkt_rate:.0f} pkt/s to overwhelm target service. "
            f"{fwd_pkts} fwd vs {bwd_pkts} bwd with {flag_str}. "
            f"Consistent with endpoint denial of service and service exhaustion."
        )
    if iat_mean > 100000 and bwd_pkts > 0 and pkt_len < 200 and fwd_bwd_ratio < 3:
        return (
            f"Botnet C2 beaconing via {port_desc}. "
            f"Compromised host communicating with C2 server via application layer protocol. "
            f"Periodic {pkt_len:.1f} byte packets every {iat_mean/1000:.0f}ms "
            f"bidirectional: {fwd_pkts} sent, {bwd_pkts} received with {flag_str}. "
            f"Consistent with bot malware maintaining persistent C2 channel."
        )
    if pkt_len > 800 and duration > 10 and flow_pkts > 100:
        return (
            f"Remote service exploitation targeting {port_desc}. "
            f"Large {pkt_len:.0f} byte payloads over {duration:.1f}s session "
            f"exploiting vulnerability for unauthorized access. "
            f"{fwd_pkts} fwd and {bwd_pkts} bwd packets with {flag_str}. "
            f"Consistent with exploitation of remote services."
        )
    if bwd_pkts > fwd_pkts and duration > 5 and dst_port not in [80,443,8080]:
        return (
            f"Lateral movement via {port_desc}. "
            f"Adversary transferring tools across network internally. "
            f"{bwd_pkts} bwd vs {fwd_pkts} fwd packets over {duration:.2f}s. "
            f"Consistent with lateral tool transfer and internal spreading."
        )
    if dst_port in [80, 443, 8080] and psh > 0:
        if pkt_len < 100 and flow_pkts > 5:
            return (
                f"Web application brute force against {port_desc}. "
                f"Adversary repeatedly submitting login attempts. "
                f"{flow_pkts} short {pkt_len:.0f} byte requests over {duration:.2f}s. "
                f"Consistent with web login brute forcing and exploitation of "
                f"public-facing application."
            )
        return (
            f"Web application attack on {port_desc} with malicious payloads. "
            f"SQL injection or XSS exploiting public-facing application vulnerabilities. "
            f"{fwd_pkts} requests, {pkt_len:.0f} byte payloads, "
            f"{duration:.2f}s with {flag_str}. "
            f"Consistent with exploit public-facing application."
        )
    if down_up > 5 and byte_rate > 5000:
        return (
            f"Data exfiltration via {port_desc}. "
            f"High outbound transfer {byte_rate:.0f} bytes/sec, "
            f"down/up ratio {down_up:.1f}. "
            f"Consistent with automated data theft and exfiltration."
        )
    return (
        f"Network flow to {port_desc} with {flag_str}. "
        f"{fwd_pkts} fwd / {bwd_pkts} bwd packets at "
        f"{pkt_rate:.1f} pkt/s, {pkt_len:.1f} byte mean size, "
        f"{duration:.2f}s duration."
    )

print("✅ Alert text generator defined!")

✅ Alert text generator defined!


In [5]:
print("⏳ Generating alert texts...")
df['alert_text'] = df.progress_apply(generate_alert_text, axis=1)
gc.collect()
print(f"✅ Done! {len(df):,} alert texts generated")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Generating alert texts...


100%|██████████████████████████████████| 62143/62143 [00:01<00:00, 61114.37it/s]


✅ Done! 62,143 alert texts generated
💾 RAM available: 10.7 GB


In [6]:
MITRE_PATH = DATA_DIR + 'enterprise-attack.json'

print("⏳ Loading MITRE ATT&CK data...")
with open(MITRE_PATH, 'r') as f:
    mitre_data = json.load(f)

techniques = []
for obj in mitre_data['objects']:
    if obj.get('type') != 'attack-pattern': continue
    if obj.get('revoked', False) or obj.get('x_mitre_deprecated', False): continue
    tech_id = next((r['external_id'] for r in obj.get('external_references', [])
                    if r.get('source_name') == 'mitre-attack'), None)
    if not tech_id: continue
    tactics = [p['phase_name'] for p in obj.get('kill_chain_phases', [])
               if p['kill_chain_name'] == 'mitre-attack']
    techniques.append({
        'technique_id':   tech_id,
        'technique_name': obj.get('name', ''),
        'description':    obj.get('description', ''),
        'tactics':        ', '.join(tactics),
        'platforms':      ', '.join(obj.get('x_mitre_platforms', [])),
    })

print(f"✅ {len(techniques)} techniques loaded")
del mitre_data
gc.collect()

print("⏳ Loading embedding model...")
emb_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")

print("⏳ Embedding MITRE techniques...")
technique_texts = [
    f"Technique {t['technique_id']}: {t['technique_name']}. "
    f"Tactics: {t['tactics']}. "
    f"Description: {t['description'][:500]}"
    for t in techniques
]
technique_embeddings = emb_model.encode(
    technique_texts, batch_size=64,
    show_progress_bar=True, convert_to_numpy=True
)
faiss.normalize_L2(technique_embeddings)

dim   = technique_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(technique_embeddings)

print(f"✅ FAISS index built: {index.ntotal} techniques indexed")

with open(DATA_DIR + 'mitre_techniques.pkl', 'wb') as f:
    pickle.dump(techniques, f)
np.save(DATA_DIR + 'mitre_embeddings.npy', technique_embeddings)
faiss.write_index(index, DATA_DIR + 'mitre_faiss.index')
print("✅ MITRE assets saved!")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Loading MITRE ATT&CK data...
✅ 691 techniques loaded
⏳ Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!
💾 RAM available: 10.5 GB
⏳ Embedding MITRE techniques...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

✅ FAISS index built: 691 techniques indexed
✅ MITRE assets saved!
💾 RAM available: 10.2 GB


In [7]:
print("⏳ Preparing training data...")
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
le = LabelEncoder()
y  = le.fit_transform(df['tactic_label'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"✅ Tactics: {list(le.classes_)}")

print("\n⏳ Training Random Forest...")
clf = RandomForestClassifier(
    n_estimators=100, max_depth=20,
    min_samples_leaf=5, n_jobs=-1,
    random_state=42, class_weight='balanced'
)
clf.fit(X_train, y_train)
gc.collect()
print("✅ Training complete!")

y_pred = clf.predict(X_test)
print("\n📊 TACTIC CLASSIFICATION REPORT:")
print("=" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_, digits=3
))

with open(DATA_DIR + 'tactic_classifier.pkl', 'wb') as f:
    pickle.dump({'model': clf, 'encoder': le, 'features': feature_cols}, f)
print("✅ Classifier saved!")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Preparing training data...
✅ Train: 49,714  Test: 12,429
✅ Tactics: ['benign', 'command-and-control', 'credential-access', 'discovery', 'impact', 'initial-access', 'lateral-movement']

⏳ Training Random Forest...
✅ Training complete!

📊 TACTIC CLASSIFICATION REPORT:
                     precision    recall  f1-score   support

             benign      0.980     0.986     0.983      1000
command-and-control      0.982     0.997     0.990       391
  credential-access      1.000     1.000     1.000      1600
          discovery      0.999     0.997     0.998      2000
             impact      0.999     0.997     0.998      7000
     initial-access      0.966     0.995     0.981       431
   lateral-movement      1.000     0.714     0.833         7

           accuracy                          0.996     12429
          macro avg      0.990     0.955     0.969     12429
       weighted avg      0.996     0.996     0.996     12429

✅ Classifier saved!
💾 RAM available: 10.2 GB


In [8]:
print("⏳ Embedding alert texts...")
alert_embeddings = emb_model.encode(
    df['alert_text'].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)
faiss.normalize_L2(alert_embeddings)
gc.collect()
print(f"✅ Alert embeddings: {alert_embeddings.shape}")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")

print("\n⏳ Running hybrid MITRE matching...")
X_all        = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
pred_tactics = le.inverse_transform(clf.predict(X_all))
del X_all
gc.collect()

p1_ids,p1_names,p1_tacs,p1_confs = [],[],[],[]
p2_ids,p2_names,p2_tacs,p2_confs = [],[],[],[]
p3_ids,p3_names,p3_tacs,p3_confs = [],[],[],[]

for i in tqdm(range(len(df)), desc="Matching"):
    tactic     = pred_tactics[i]
    tac_idxs   = [j for j,t in enumerate(techniques) if tactic in t['tactics']]

    if len(tac_idxs) >= 3:
        sub_embs  = technique_embeddings[tac_idxs]
        sims      = (alert_embeddings[i:i+1] @ sub_embs.T)[0]
        top3_loc  = np.argsort(sims)[::-1][:3]
        top3_idx  = [tac_idxs[k] for k in top3_loc]
        top3_sims = sims[top3_loc]
    else:
        s, idx    = index.search(alert_embeddings[i:i+1], 3)
        top3_idx  = idx[0].tolist()
        top3_sims = s[0]

    for rank, (gidx, sim) in enumerate(zip(top3_idx, top3_sims)):
        t = techniques[gidx]
        [p1_ids,p2_ids,p3_ids][rank].append(t['technique_id'])
        [p1_names,p2_names,p3_names][rank].append(t['technique_name'])
        [p1_tacs,p2_tacs,p3_tacs][rank].append(t['tactics'])
        [p1_confs,p2_confs,p3_confs][rank].append(round(float(sim), 4))

df['ml_pred_tactic']   = pred_tactics
df['pred_technique_1'] = p1_ids;  df['pred_tech_name_1'] = p1_names
df['pred_tactic_1']    = p1_tacs; df['pred_confidence_1'] = p1_confs
df['pred_technique_2'] = p2_ids;  df['pred_tech_name_2'] = p2_names
df['pred_tactic_2']    = p2_tacs; df['pred_confidence_2'] = p2_confs
df['pred_technique_3'] = p3_ids;  df['pred_tech_name_3'] = p3_names
df['pred_tactic_3']    = p3_tacs; df['pred_confidence_3'] = p3_confs
gc.collect()
print("✅ Hybrid matching complete!")

⏳ Embedding alert texts...


Batches:   0%|          | 0/486 [00:00<?, ?it/s]

✅ Alert embeddings: (62143, 384)
💾 RAM available: 10.0 GB

⏳ Running hybrid MITRE matching...


Matching: 100%|████████████████████████| 62143/62143 [00:03<00:00, 18103.48it/s]


✅ Hybrid matching complete!


In [9]:
def get_parent_id(tid):
    return tid.split('.')[0] if isinstance(tid, str) and '.' in tid else tid

attack_df = df[df['Label'] != 'BENIGN'].copy()
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)
attack_df['pred1_parent'] = attack_df['pred_technique_1'].apply(get_parent_id)
attack_df['pred2_parent'] = attack_df['pred_technique_2'].apply(get_parent_id)
attack_df['pred3_parent'] = attack_df['pred_technique_3'].apply(get_parent_id)

top1 = (attack_df['true_parent'] == attack_df['pred1_parent']).mean()
top3 = attack_df.apply(
    lambda r: r['true_parent'] in [
        r['pred1_parent'], r['pred2_parent'], r['pred3_parent']
    ], axis=1
).mean()

print("📊 HYBRID SYSTEM — FINAL RESULTS")
print("=" * 58)
print(f"   Top-1 Accuracy: {top1:.2%}")
print(f"   Top-3 Accuracy: {top3:.2%}")
print()
print(f"{'Attack Type':<35} {'Top-1':>6}  {'Top-3':>6}  {'Count':>7}")
print("-" * 60)
for label in sorted(attack_df['Label'].unique()):
    sub = attack_df[attack_df['Label'] == label]
    t1  = (sub['true_parent'] == sub['pred1_parent']).mean()
    t3  = sub.apply(lambda r: r['true_parent'] in [
              r['pred1_parent'], r['pred2_parent'],
              r['pred3_parent']], axis=1).mean()
    status = "✅" if t3 >= 0.5 else "⚠️ "
    print(f"   {status} {label:<33} {t1:>5.0%}   {t3:>5.0%}   {len(sub):>6,}")

📊 HYBRID SYSTEM — FINAL RESULTS
   Top-1 Accuracy: 66.34%
   Top-3 Accuracy: 83.45%

Attack Type                          Top-1   Top-3    Count
------------------------------------------------------------
   ⚠️  Bot                                  1%     39%    1,953
   ✅ DDoS                                 0%     54%   10,000
   ✅ DoS GoldenEye                       97%     99%    5,000
   ✅ DoS Hulk                            97%    100%   10,000
   ✅ DoS Slowhttptest                    76%     81%    5,000
   ✅ DoS slowloris                       69%     87%    5,000
   ✅ FTP-Patator                         68%     68%    5,000
   ⚠️  Heartbleed                           0%      0%       11
   ⚠️  Infiltration                        25%     33%       36
   ✅ PortScan                            98%     99%   10,000
   ✅ SSH-Patator                         93%     93%    3,000
   ✅ Web Attack - Brute Force             7%    100%    1,470
   ✅ Web Attack - SQL Injection          43%

In [12]:
OUT = DATA_DIR + 'final_mitre_mapped.csv'
df.to_csv(OUT, index=False)
print(f"✅ Final dataset saved → {OUT}")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 RAM available: {psutil.virtual_memory().available/1024**3:.1f} GB")
print("\n🎉 Phase 4 Complete!")
print("   Next → Phase 5: Risk Scoring Engine")


✅ Final dataset saved → /home/arshad/Network-project/data/final_mitre_mapped.csv
📊 Shape: 62,143 rows × 45 columns
💾 RAM available: 9.3 GB

🎉 Phase 4 Complete!
   Next → Phase 5: Risk Scoring Engine
